# Automotive Data Mapper - MVP: Exploratory Data Analysis (EDA)

Date: 2026-08-06  
Author: Luis Renteria Lezano  
[LinkedIn](https://www.linkedin.com/in/renteria-luis) | [GitHub](https://github.com/renteria-luis) | [Portfolio](https://luisrenteria.me)

## Executive Summary

- Goal: explore the raw automotive service-record feeds before mapping them into one canonical schema, starting with the data types and formatting problems found in the Shop A feed. This notebook is the first step of the exploratory phase of a vehicle history record (VHR) style data-mapping project.
- **Sources:** Three synthetic vehicle service feeds representing an **independent repair shop**, a **dealership management system**, and a **fleet maintenance provider**:
  - `shop_a`: CSV
  - `dealer_b`: XML
  - `fleet_c`: JSON
- **Data:** [`../data/raw/`](https://github.com/renteria-luis/automotive-data-mapper/tree/main/data/raw)
- **Data dictionary:** [`../docs/DATA_DICTIONARY.md`](https://github.com/renteria-luis/automotive-data-mapper/blob/main/docs/DATA_DICTIONARY.md)
- **Sample data documentation:** [`../docs/SAMPLE_DATA.md`](https://github.com/renteria-luis/automotive-data-mapper/blob/main/docs/SAMPLE_DATA.md)

## 0. Reproducibility & Environment Setup

Imports, routes, versions.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import csv

import pandas as pd
from src.profiling import profile


ROOT = Path('..')

RAW = ROOT / "data" / "raw"
SHOP_A = RAW / "shop_a" / "service_records_20260731.csv"
DEALER_B = RAW / "dealer_b" / "ProcessRepairOrder_20260731.xml"
FLEET_C = RAW / "fleet_c" / "maintenance_events_2026-07.json"

for p in (SHOP_A, DEALER_B, FLEET_C):
    print(p.exists(), p)

True ../data/raw/shop_a/service_records_20260731.csv
True ../data/raw/dealer_b/ProcessRepairOrder_20260731.xml
True ../data/raw/fleet_c/maintenance_events_2026-07.json


## 1. An independent auto repair shop: `shop_a` - CSV
42 rows, 24 columns. 

### 1.1 Raw look
Before reading it with pandas, inspect the file as plain text. This is not just ceremony: it confirms four things that `read_csv` takes for granted and that, if they are wrong, can cause it to fail **without warning**.

* What the separator is
* How values containing that separator are marked
* How lines end
* Whether the file starts with invisible characters

Using `repr()` instead of `print()` reveals characters that are normally invisible.

In [2]:
with open(SHOP_A, encoding="utf-8") as f:
    head = [next(f) for _ in range(3)]

for line in head:
    print(repr(line))

'VIN,RO_OPEN_DATE,RO_CLOSE_DATE,MILEAGE,ODOMETER_MEASURE,RO_INVOICE_NUMBER,SERVICE_DESCRIPTION,LABOR_DESCRIPTION,PART_NAME_DESCRIPTION,PART_QUANTITY,MAKE,MODEL,MODEL_YEAR,PLATE,PLATE_STATE,MANAGEMENT_SYSTEM,LOCATION_ID,LOCATION_NAME,ADDRESS,CITY,STATE,POSTAL_CODE,PHONE,URL\n'
'1FTFW1E50KFA12345,10/10/2019,10/10/2019,"21,000",KM,184200,"Lube oil and filter, 5W30 synthetic","LUBE OIL AND FILTER, 5W30 SYNTHETIC",OIL FILTER,1,FORD,F-150,2019,CJKT 421,ON,PROTRACTOR,CA-ON-4471,Riverside Auto Service,1247 Hamilton Rd,London,ON,N5W 1A7,5194553120,www.riversideautoservice.ca\n'
'1FTFW1E50KFA12345,06/03/2020,06/04/2020,30733,KM,184203,Replace front brake pads and machine rotors,REPLACE FRONT BRAKE PADS AND MACHINE ROT,BRAKE PAD SET,2,FORD,F-150,2019,,,PROTRACTOR,CA-ON-4471,RIVERSIDE AUTO SERVICE,1247 Hamilton Rd,London,ON,N5W 1A7,5194553120,www.riversideautoservice.ca\n'


- [X] It starts directly with `b'1FTW...`, which means there is no BOM.
- [X] The line ending is `\r\n`.
- [X] Commas are used as the delimiter.
- [X] Quotes are used for quoted fields. They are applied when the text contains a comma, which prevents the parser from splitting that record.
- [X] All data appears to be ASCII, so UTF-8 should be valid. This will be verified in the next cell.

In [3]:
row = head[1]
print('splitting by commas:', len(row.split(',')), 'fields')
print('parsing as CSV:', len(next(csv.reader([row]))), 'fields')
print('fields in the header:', len(next(csv.reader([head[0]]))))

splitting by commas: 27 fields
parsing as CSV: 24 fields
fields in the header: 24


The first record shows that `MILEAGE` is `21,000`. Because it contains a comma, it is parsed as a `str`, and when the data is split by commas, the first row ends up with 25 fields instead of 24.

### 1.2 Data Reading

`shop_a_raw` is read once and **never modified again**. Any transformation is performed on a copy.

Without this rule, after three cells you can no longer compare the before and after, and the notebook can no longer be run from top to bottom.

In [4]:
shop_a_raw = pd.read_csv(SHOP_A)
shop_a = shop_a_raw.copy()

print(shop_a_raw.shape)
shop_a_raw.head(2)

(42, 24)


,VIN,RO_OPEN_DATE,RO_CLOSE_DATE,MILEAGE,ODOMETER_MEASURE,RO_INVOICE_NUMBER,SERVICE_DESCRIPTION,LABOR_DESCRIPTION,PART_NAME_DESCRIPTION,PART_QUANTITY,...,PLATE_STATE,MANAGEMENT_SYSTEM,LOCATION_ID,LOCATION_NAME,ADDRESS,CITY,STATE,POSTAL_CODE,PHONE,URL
0,1FTFW1E50KFA12345,10/10/2019,10/10/2019,"21,000",KM,184200,"Lube oil and filter, 5W30 synthetic","LUBE OIL AND FILTER, 5W30 SYNTHETIC",OIL FILTER,1.0,...,ON,PROTRACTOR,CA-ON-4471,Riverside Auto Service,1247 Hamilton Rd,London,ON,N5W 1A7,5194553120,www.riversideautoservice.ca
1,1FTFW1E50KFA12345,06/03/2020,06/04/2020,30733,KM,184203,Replace front brake pads and machine rotors,REPLACE FRONT BRAKE PADS AND MACHINE ROT,BRAKE PAD SET,2.0,...,NaN,PROTRACTOR,CA-ON-4471,RIVERSIDE AUTO SERVICE,1247 Hamilton Rd,London,ON,N5W 1A7,5194553120,www.riversideautoservice.ca


### 1.3 Data Types

`read_csv` reads everything as text and then tries to convert each column to a number. If **a single value** in the column cannot be converted, it cancels the conversion for the entire column and leaves it as `object`. If there are `NaN` they are typed as `float`.

In [5]:
shop_a_raw.dtypes

VIN                       object
RO_OPEN_DATE              object
RO_CLOSE_DATE             object
MILEAGE                   object
ODOMETER_MEASURE          object
RO_INVOICE_NUMBER          int64
SERVICE_DESCRIPTION       object
LABOR_DESCRIPTION         object
PART_NAME_DESCRIPTION     object
PART_QUANTITY            float64
MAKE                      object
MODEL                     object
MODEL_YEAR               float64
PLATE                     object
PLATE_STATE               object
MANAGEMENT_SYSTEM         object
LOCATION_ID               object
LOCATION_NAME             object
ADDRESS                   object
CITY                      object
STATE                     object
POSTAL_CODE               object
PHONE                      int64
URL                       object
dtype: object

> For example, the fact that the data type of `MODEL_YEAR` is `float` indicates that there is at least one year recorded as a float or NaN.

### 1.4 What `object` dtype does not tell

`object` means pandas stopped making assumptions about that column. There could be one type or five, and the dtype looks the same. To know what is actually inside, you have to count the real data types.

In [6]:
print(shop_a_raw['MILEAGE'].map(type).value_counts())
print()
print('first values:', shop_a_raw['MILEAGE'].head(3).tolist())

MILEAGE
<class 'str'>      41
<class 'float'>     1
Name: count, dtype: int64

first values: ['21,000', '30733', '33255']


By counting the types of values in `MILEAGE`, we confirm that `read_csv` converted everything to `str`, with the only `float` being the `NaN`, as mentioned earlier. This shows that simply using `.dtype` does not tell us the actual type of a specific value, which is why we will use **Pydantic**.

### 1.5 Mapping dictionary

Here we declare the **source-to-target mapping (STTM)**: each source column names its target field, or `None` when it has no target.

It is written as data rather than logic for three reasons:

* Adding a column to the feed means adding one line here, not editing code.
* Mapping coverage can be counted and audited.
* Columns with `None` are exactly what the `E010` code counts.

The values come from `docs/DATA_DICTIONARY.md`. The dictionary does not invent anything; it implements it.

In [7]:
SHOP_A_MAP: dict[str, str | None] = {
    "VIN": "vin",
    "RO_OPEN_DATE": "event_date",
    "RO_CLOSE_DATE": None,
    "MILEAGE": "odometer_km",
    "ODOMETER_MEASURE": "odometer_source_unit",
    "RO_INVOICE_NUMBER": "source_record_id",
    "SERVICE_DESCRIPTION": "raw_description",
    "LABOR_DESCRIPTION": None,
    "PART_NAME_DESCRIPTION": None,
    "PART_QUANTITY": None,
    "MAKE": None,
    "MODEL": None,
    "MODEL_YEAR": None,
    "PLATE": None,
    "PLATE_STATE": None,
    "MANAGEMENT_SYSTEM": None,
    "LOCATION_ID": None,
    "LOCATION_NAME": "provider_name",
    "ADDRESS": None,
    "CITY": "provider_city",
    "STATE": "provider_province",
    "POSTAL_CODE": None,
    "PHONE": None,
    "URL": None,
}

#### Coverage audit

This check verifies that the columns in the file and the columns in the dictionary match.

If the file contains a column that the dictionary does not know about, or the dictionary mentions a column that no longer exists in the file, there is a problem with the mapping. Without this audit, the pipeline could keep running without showing any errors, and that column would simply disappear.

That is why this check helps detect incomplete or outdated mappings and protect data integrity.

This is literally *auditing mappings for data integrity*.

In [8]:
in_file = set(shop_a_raw.columns)
in_map = set(SHOP_A_MAP)

print('in the file and not in the dictionary:', in_file - in_map)
print('in the dictionary and not in the file:', in_map - in_file)
assert in_file == in_map, 'the dictionary and the file do not match'

mapped = [c for c, f in SHOP_A_MAP.items() if f is not None]
unmapped = [c for c, f in SHOP_A_MAP.items() if f is None]

print(f'\n{len(mapped)} mapped | {len(unmapped)} without target | coverage {len(mapped) / len(SHOP_A_MAP):.0%}')
print('\nwithout target (E010):', unmapped)

in the file and not in the dictionary: set()
in the dictionary and not in the file: set()

9 mapped | 15 without target | coverage 38%

without target (E010): ['RO_CLOSE_DATE', 'LABOR_DESCRIPTION', 'PART_NAME_DESCRIPTION', 'PART_QUANTITY', 'MAKE', 'MODEL', 'MODEL_YEAR', 'PLATE', 'PLATE_STATE', 'MANAGEMENT_SYSTEM', 'LOCATION_ID', 'ADDRESS', 'POSTAL_CODE', 'PHONE', 'URL']


#### Why the 15 columns without a target are not garbage

`MAKE`, `MODEL`, `MODEL_YEAR`, and `PLATE` could be used to match records for the same vehicle across feeds, and to verify that a VIN decodes to the correct vehicle. `POSTAL_CODE` and `ADDRESS` could be used to identify the shop when its name is written in different ways.

Reporting them as `E010` instead of ignoring them turns that gap into a **documented decision** rather than an oversight.

### 1.6 Profiling of columns that have a target

**Profiling:** discover/explore -> What did I find in the data, and what does it imply?

**Validation:** decision. -> What conditions must a record meet to pass?

Example:

- VIN contains `NaN` -> **Profiling:** I discovered it.
- VIN with `NaN` is rejected -> **Validation:** I made a decision.
- MI and KM -> **Profiling:** I discovered two different units in one column.
- Converting MI to KM -> **Transformation:** I decided how to normalize them.

In [9]:
profile(shop_a_raw, mapped)

,column,nulls,unique,spare_spaces
0,VIN,0,26,0.0
1,RO_OPEN_DATE,0,39,0.0
2,MILEAGE,1,40,0.0
3,ODOMETER_MEASURE,0,2,0.0
4,RO_INVOICE_NUMBER,0,41,NaN
5,SERVICE_DESCRIPTION,1,33,8.0
6,LOCATION_NAME,0,3,0.0
7,CITY,0,1,0.0
8,STATE,0,1,0.0


#### Value inventory

Counting nulls is not enough for columns that determine code branches. So need to inspect the actual values.

In [10]:
for c in ['ODOMETER_MEASURE', 'LOCATION_NAME', 'CITY', 'STATE']:
    values = sorted(shop_a_raw[c].dropna().unique().tolist())
    print(f'{c:20} {len(values)} distinct -> {values}')

print('\nrows in miles:', int((shop_a_raw['ODOMETER_MEASURE'] == 'MI').sum()))

ODOMETER_MEASURE     2 distinct -> ['KM', 'MI']
LOCATION_NAME        3 distinct -> ['RIVERSIDE AUTO SERVICE', 'Riverside Auto Service', 'Riverside Auto Service Ltd']
CITY                 1 distinct -> ['London']
STATE                1 distinct -> ['ON']

rows in miles: 6


### 1.7 Duplicates within the feed

There are two levels of deduplication, and this is the cheap one: **an identical row within the same file**.

It can be detected without mapping anything. The expensive case—when the same event is represented differently or arrives through two feeds—requires the canonical schema and belongs in [`02_pipeline.ipynb`](02_pipeline.ipynb).

In [12]:
print('exact duplicate rows:', int(shop_a_raw.duplicated().sum()))
print('unique RO_INVOICE_NUMBER:', shop_a_raw['RO_INVOICE_NUMBER'].nunique(), 'of', len(shop_a_raw))

repeated = shop_a_raw[shop_a_raw['RO_INVOICE_NUMBER'].duplicated(keep=False)]
repeated[['RO_INVOICE_NUMBER', 'VIN', 'RO_OPEN_DATE', 'MILEAGE', 'SERVICE_DESCRIPTION', 'LABOR_DESCRIPTION']]

exact duplicate rows: 1
unique RO_INVOICE_NUMBER: 41 of 42


,RO_INVOICE_NUMBER,VIN,RO_OPEN_DATE,MILEAGE,SERVICE_DESCRIPTION,LABOR_DESCRIPTION
34,184302,1FA6P8TH2J5142608,01/29/2019,65220,Weld exhaust flex pipe,WELD EXHAUST FLEX PIPE
35,184302,1FA6P8TH2J5142608,01/29/2019,65220,Weld exhaust flex pipe,WELD EXHAUST FLEX PIPE


### 1.8 What the profiling forces us to write
Every rule below is justified by a finding from the previous cells, not by intuition.
* **`MILEAGE`**
  * Finding: 1 null, thousands separator, all values are strings.
  * Rule: remove the separator, use a nullable integer, convert when the unit is `MI`, and treat empty values and `0` as unknown.  
* **`ODOMETER_MEASURE`**
  * Finding: 2 values, no nulls.
  * Rule: convert to lowercase and use it to determine how `MILEAGE` should be converted.
* **`RO_INVOICE_NUMBER`**
  * Finding: 41 unique values out of 42 rows.
  * Rule: detect the exact duplicate and reject the second occurrence as `E009`.
* **`SERVICE_DESCRIPTION`**
  * Finding: 1 null, 8 values with extra spaces.
  * Rule: apply `strip`; an empty description becomes `E004`.
* **`LOCATION_NAME`**
  * Finding: 3 different spellings for the same shop.
  * Rule: standardize them to one form.
* **`RO_OPEN_DATE`**
  * Finding: no nulls, `MM/DD/YYYY` format.
  * Rule: parse using an explicit format; an invalid date becomes `E005`.
* **`VIN`**, **`CITY`**, **`STATE`**
  * Finding: clean.
  * Rule: copy them, with defensive `strip` and `upper`.

**Design decision from the profiling:**
`LOCATION_NAME` has 3 distinct values, while `LOCATION_ID` has only 1. The identifier is stable; the name is not. Therefore, the provider is identified by `LOCATION_ID`, while the name remains presentation text.

This decision goes into [`../docs/DESIGN_DECISIONS.md`](../docs/DESIGN_DECISIONS.md), together with the rejected alternative.
